# 01 — Daten: VisDrone-DET 2019 vorbereiten

Download, COCO-Konvertierung, SAHI-Slicing (640x640 @ 20% Overlap),
Statistik und auto-generierter Report nach `docs/data_report.md`.

In [ ]:
# Zelle 1 — Setup: Dependencies + Drive
!pip install -q -r requirements.txt
!pip install -q gdown

from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/ba-mamba-neck'
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
# Zelle 2 — VisDrone downloaden, konvertieren, slicen
!python data/prepare.py --output /content/visdrone

In [ ]:
# Zelle 3 — 5 Beispielbilder aus dem Val-Split mit Bounding Boxes
import json, random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

ROOT = Path('/content/visdrone')
VAL_JSON = ROOT / 'annotations' / 'val_unsliced.json'
VAL_IMAGES = ROOT / 'raw' / 'VisDrone2019-DET-val' / 'images'

coco = json.loads(VAL_JSON.read_text())
id_to_name = {c['id']: c['name'] for c in coco['categories']}
ann_by_img = {}
for a in coco['annotations']:
    ann_by_img.setdefault(a['image_id'], []).append(a)

random.seed(42)
samples = random.sample(coco['images'], k=5)

fig, axes = plt.subplots(5, 1, figsize=(12, 30))
for ax, img_info in zip(axes, samples):
    img = Image.open(VAL_IMAGES / img_info['file_name'])
    ax.imshow(img)
    for ann in ann_by_img.get(img_info['id'], []):
        x, y, w, h = ann['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=1,
                                       edgecolor='lime', facecolor='none'))
        ax.text(x, max(0, y - 2), id_to_name[ann['category_id']],
                color='lime', fontsize=6)
    ax.set_title(f"{img_info['file_name']} — {len(ann_by_img.get(img_info['id'], []))} objs")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 4 — Größenverteilungs-Histogramm (COCO-Buckets)
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('/content/visdrone')
splits = {
    'train/unsliced': ROOT / 'annotations' / 'train_unsliced.json',
    'val/unsliced':   ROOT / 'annotations' / 'val_unsliced.json',
    'train/sliced':   ROOT / 'annotations' / 'train_sliced.json',
    'val/sliced':     ROOT / 'annotations' / 'val_sliced.json',
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (label, path) in zip(axes.flat, splits.items()):
    coco = json.loads(path.read_text())
    areas = np.array([a['area'] for a in coco['annotations']])
    sides = np.sqrt(areas)
    ax.hist(sides, bins=60, color='steelblue', edgecolor='white')
    ax.axvline(32, color='orange', ls='--', label='small/medium (32)')
    ax.axvline(96, color='red', ls='--', label='medium/large (96)')
    ax.set_title(f'{label} — n={len(areas)}')
    ax.set_xlabel('sqrt(area) [px]')
    ax.set_ylabel('#annotations')
    ax.set_xlim(0, 300)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 5 — docs/data_report.md auto-generieren
import json
from pathlib import Path

ROOT = Path('/content/visdrone')
REPORT = Path('docs/data_report.md')
summary = json.loads((ROOT / 'summary.json').read_text())

def fmt_buckets(s):
    b = s['size_buckets']
    p = s['size_buckets_pct']
    return (f"small={b['small']} ({p['small']}%)  "
            f"medium={b['medium']} ({p['medium']}%)  "
            f"large={b['large']} ({p['large']}%)")

lines = [
    '# Data Report — VisDrone-DET 2019',
    '',
    '_Auto-generiert von `notebooks/01_data.ipynb` (P1)._',
    '',
    '## Pipeline',
    '',
    '1. Download VisDrone-DET train + val (official GDrive).',
    '2. TXT → COCO: drop Kategorie 0 (ignored) und 11 (others); 10 Klassen bleiben.',
    '3. SAHI-Slicing: 640x640 @ 20% Overlap (train + val). Val-Unsliced bleibt für SAHI-Inference.',
    '',
    '## Bild- und Annotationszahlen',
    '',
    '| Split | #Images | #Annotations | Größenverteilung (COCO) |',
    '|---|---:|---:|---|',
]
for split in ('train', 'val'):
    for kind in ('unsliced', 'sliced'):
        s = summary[split][kind]
        lines.append(
            f"| {split}/{kind} | {s['num_images']} | {s['num_annotations']} | {fmt_buckets(s)} |"
        )

lines += ['', '## Klassenverteilung (unsliced)', '',
          '| Klasse | Train | Val |',
          '|---|---:|---:|']
train_cls = summary['train']['unsliced']['classes']
val_cls = summary['val']['unsliced']['classes']
for name in train_cls:
    lines.append(f"| {name} | {train_cls[name]} | {val_cls.get(name, 0)} |")

lines += ['', '## Slicing-Statistik', '',
          f"- Tile-Size: 640x640",
          f"- Overlap: 20%",
          f"- Train sliced: {summary['train']['sliced']['num_images']} Tiles, "
          f"{summary['train']['sliced']['num_annotations']} Annotationen",
          f"- Val sliced:   {summary['val']['sliced']['num_images']} Tiles, "
          f"{summary['val']['sliced']['num_annotations']} Annotationen",
          '']

REPORT.parent.mkdir(exist_ok=True)
REPORT.write_text('\n'.join(lines))
print(f'wrote {REPORT} ({REPORT.stat().st_size} bytes)')